In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import time
import math
import statistics
import random
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from torch_geometric.nn import GCNConv
from scipy.sparse import coo_matrix
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
import evaluate # For BLEU calculation
import nltk.translate.bleu_score as bleu
from nltk.tokenize import word_tokenize
# Note: You may need to run 'import nltk; nltk.download('punkt')' separately

In [2]:
# --- Constants & Global Setup ---
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Tokenizer and IDs ---
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 77
NUM_CATEGORIES = 53
NUM_OBJECTS = 61
D_MODEL = 256 # Transformer hidden dimension

# --- Granger Causality Matrix Creation ---
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j: continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            data = np.vstack([ts_j, ts_i]).T
            try:
                # maxlag=5 is used for the test
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                # We use the F-test result from the 5th lag
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# --- Dataset and DataLoader Components ---
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# --- Transformer Core Components ---
def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        batch_size = query.size(0)
        
        # 1) Linear projections
        query, key, value = [l(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key, value))]

        # 2) Scaled Dot Product Attention
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        p_attn = F.softmax(scores, dim=-1)
        p_attn = self.dropout(p_attn)
        
        # 3) Concatenate and final linear layer
        x = torch.matmul(p_attn, value)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, d_meta):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Metadata Conditional Bias (Semantic Gating)
        self.meta_gate = nn.Sequential(
            nn.Linear(d_meta, d_model),
            nn.Sigmoid()
        )
        
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask, meta_features):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        
        # Apply Metadata Conditional Bias
        gate = self.meta_gate(meta_features).unsqueeze(1) # (B, 1, D_model)
        x = x * gate 
        
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x

# --- Metadata Encoder ---
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_categories, num_objects, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        
        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim

    def forward(self, metadata):
        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        object_features_raw = metadata[:, 2:]
        
        object_features_raw = object_features_raw.float()

        color_vec = self.color_embedding(color_ids)
        category_vec = self.category_embedding(category_ids)
        object_vec = self.object_processor(object_features_raw)

        combined_features = torch.cat([color_vec, category_vec, object_vec], dim=1)
        return combined_features

In [3]:
# --- SpatioTemporal EEG Encoder (GCN + Transformer Encoder) ---
class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=62, d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.spatial_dropout = nn.Dropout(dropout)

        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.transformer_layers = get_clones(encoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, num_channels, num_timesteps = eeg.shape
        
        # --- Spatial Processing (GCN) ---
        batch_edge_index, batch_edge_attr = self._prepare_gcn_input(batch_size, num_timesteps, edge_index, edge_attr)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels) # (B*T, C)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.spatial_dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        
        x = x.reshape(batch_size, num_timesteps, self.d_model)
        
        # --- Transformer Input Preparation ---
        cls_token = self.cls_token.repeat(batch_size, 1, 1) # (B, 1, D_model)
        x = torch.cat([cls_token, x], dim=1) # (B, T+1, D_model)

        x = self.pos_encoding(x)

        # --- Temporal Processing (Transformer Encoder Stack) ---
        for layer in self.transformer_layers:
            x = layer(x)
        
        return self.layer_norm(x)

    def _prepare_gcn_input(self, batch_size, num_timesteps, edge_index, edge_attr):        
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=edge_index.device) * self.num_channels
        batch_edge_index += batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)
        return batch_edge_index, batch_edge_attr

# --- Transformer Decoder ---
class DecoderTF(nn.Module):
    def __init__(self, vocab_size, emb_dim, d_model, num_layers, num_heads, d_ff, pad_id, dropout, d_meta):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.pos_encoding = PositionalEncoding(emb_dim)
        self.input_projection = nn.Linear(emb_dim, d_model) 
        
        decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, d_meta)
        self.transformer_layers = get_clones(decoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, target_text_ids, memory, memory_mask, meta_features):
        tgt_embed = self.embedding(target_text_ids)
        x = self.pos_encoding(tgt_embed)
        x = self.input_projection(x)

        tgt_seq_len = target_text_ids.size(1)
        tgt_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len), diagonal=1).bool().to(x.device)
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
        
        for layer in self.transformer_layers:
            x = layer(x, memory, memory_mask, tgt_mask, meta_features)
        
        x = self.layer_norm(x)
        return self.fc_out(x)

In [4]:
class Seq2SeqTF(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, 
                 d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, pad_id=PAD_ID, dropout=0.1, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        
        self.encoder = SpatioTemporalEEGEncoderTF(
            d_model=d_model, num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, dropout=dropout
        )
        self.meta_encoder = MetadataEncoder(
            num_colors, num_categories, num_objects, color_emb_dim, category_emb_dim, object_feature_dim
        )
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = DecoderTF(
            text_vocab_size, d_model, d_model, num_layers, num_heads, d_ff, pad_id, dropout, meta_features_dim
        )
        
        self.meta_head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_categories + num_objects)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects
        self.pad_id = pad_id

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr):
        eeg_features = self.encoder(eeg, edge_index, edge_attr) # (B, T+1, D_model)
        meta_features = self.meta_encoder(metadata) # (B, D_meta)
        
        cls_token_feature = eeg_features[:, 0, :] # (B, D_model)
        meta_preds = self.meta_head(cls_token_feature)
        
        decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
        memory_mask = None 
        
        text_logits = self.decoder(
            target_text[:, :-1], # Target input (shifted right)
            decoder_memory, 
            memory_mask, 
            meta_features
        )
        
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        pred_object = meta_preds[:, self.num_colors + self.num_categories:]
        
        return text_logits, pred_color, pred_category, pred_object

In [5]:
# --- Training and Evaluation Functions ---
def train_one_epoch_tf(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training TF", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()
        
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(text_loss=text_loss.item(), meta_loss=loss.item() - text_loss.item())
        
    return total_loss / len(loader)

In [7]:
def evaluate_tf(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        total_loss += loss.item()
        
    return total_loss / len(loader)

In [9]:
@torch.no_grad()
def generate_text_beam(model, eeg_signal, meta_signal, edge_index, edge_attr, 
                       beam_width=5, max_len=64):
    """Generates text using Beam Search for the Transformer Decoder."""
    model.eval()
    
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG and Metadata (Static Memory)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)
    
    decoder_memory = eeg_features[:, 1:, :] 
    memory_mask = None 

    # 2. Initialization: Beams list stores (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 3. Beam Search Loop
    for _ in range(max_len):
        new_beams = []
        
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            input_ids = torch.tensor([seq], dtype=torch.long, device=device)

            logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
            next_token_logits = logits[:, -1, :].squeeze(0)
            
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob 
                
                new_beams.append((new_seq, new_score))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        if beams[0][0][-1] == EOS_ID:
            break
    best_seq = beams[0][0]
    
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    return predicted_text

In [10]:
# --- Execute Data Setup ---
print("--- 1. Setting up Data and Static Graph ---")
g = torch.Generator().manual_seed(42)
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
print(f"Data loaders created: Train={n_train}, Val={n_val}, Test={n_test}")

eeg_b, _, _ = next(iter(train_loader))
granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
num_channels = eeg_b.shape[1]

# Handle graph normalization and device movement
granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels)
granger_edge_index = granger_edge_index.to(torch.long).to(device)
granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
print(f"Static Granger Graph ready on {device}.")

# --- Model and Loss Initialization ---
new_model = Seq2SeqTF(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS, num_categories=NUM_CATEGORIES, num_objects=NUM_OBJECTS,
    d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, pad_id=PAD_ID, dropout=0.1
).to(device)

text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
object_criterion = nn.BCEWithLogitsLoss()

# --- Training Loop (50 epochs) ---
# NOTE: This loop will start training immediately. Interrupt if needed.
print("\n--- 2. Starting Transformer Training (Beam Search Prep) ---")

new_optimizer = AdamW(new_model.parameters(), lr=3e-5, weight_decay=1e-2)
new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=2, verbose=True)
EPOCHS = 50
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model.pt'
best_val_loss = float('inf')

print(f"Model Parameters: {sum(p.numel() for p in new_model.parameters() if p.requires_grad):,}")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_loss = train_one_epoch_tf(new_model, train_loader, new_optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    val_loss = evaluate_tf(new_model, val_loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    new_scheduler.step(val_loss)
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
    
    print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
    print(f"\tTrain Loss: {train_loss:.4f}")
    print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(new_model.state_dict(), MODEL_SAVE_PATH)
        print("\t-> Validation loss improved, saving new best Transformer model. 🏆")

print("\n--- Training Complete. Starting Final Evaluation ---")



--- 1. Setting up Data and Static Graph ---
Data loaders created: Train=22400, Val=2800, Test=2800


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Static Granger Graph ready on cuda.

--- 2. Starting Transformer Training (Beam Search Prep) ---
Model Parameters: 23,527,913


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

...


Epoch 10/50 | Time: 02m 37s
	Train Loss: 1.6561
	 Val. Loss: 1.5278 | Val. Perplexity:  4.6079
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 11/50 | Time: 02m 37s
	Train Loss: 1.5407
	 Val. Loss: 1.4146 | Val. Perplexity:  4.1149
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [12]:
# --- 3. Final Beam Search Evaluation ---
print("\n--- Final Beam Search & BLEU Evaluation ---")

# Load the best weights
new_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))

predictions_list = []
references_list = []
samples_printed = 0
NUM_SAMPLES_TO_PRINT = 10 
global_sample_index = 0

bleu_metric = evaluate.load('bleu')
test_progress_bar = tqdm(test_loader, desc="Beam Search Evaluation", leave=True)
start_time = time.time()

for eeg_b, meta_b, txt_b in test_progress_bar:
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i]
        true_text_ids = txt_b[i]
        
        # --- Generation using BEAM SEARCH ---
        predicted_text = generate_text_beam(
            new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr, beam_width=5
        )
        
        # --- Ground Truth Text ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        
        if not true_text: continue
            
        # Store for Corpus-Level Evaluation
        predictions_list.append(predicted_text)
        references_list.append([true_text]) # Hugging Face 'evaluate' expects list of lists for references

        # --- Print Sample Analysis (Qualitative) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            print(f"\n--- Sample {global_sample_index + 1} ---")
            print(f"GROUND TRUTH: {true_text}")
            print(f"PREDICTION:   {predicted_text}")
            samples_printed += 1
            
        global_sample_index += 1

# --- Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Beam Search Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score (Beam 5):** {final_bleu_score:.4f} |")
print("=============================================")


Beam Search Evaluation Complete in 09m 46s
| Total Samples Evaluated:  2800 |
---------------------------------------------
| **Corpus-Level BLEU Score (Beam 5):** 0.2519 |


In [1]:
import torch
import torch.nn.functional as F
import statistics
from tqdm.auto import tqdm
import time
import math
import evaluate 
import json 
import copy # Added in case other functions need it

# --- 0. Metadata Mapping Restoration ---
@torch.no_grad()
# def generate_text_beam(model, eeg_signal, meta_signal, edge_index, edge_attr, 
#                        beam_width=5, max_len=64):
#     """Generates text using Beam Search for the Transformer Decoder."""
#     model.eval()
    
#     # Unsqueeze input to create a batch size of 1
#     eeg_signal = eeg_signal.unsqueeze(0).to(device)
#     meta_signal = meta_signal.unsqueeze(0).to(device)
    
#     # 1. Encode EEG and Metadata (Static Memory)
#     eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
#     meta_features = model.meta_encoder(meta_signal)
    
#     decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
#     memory_mask = None 

#     # 2. Predict Metadata from CLS token (Needed for printing results)
#     cls_token_feature = eeg_features[:, 0, :]
#     meta_preds = model.meta_head(cls_token_feature).squeeze(0) 
#     pred_color_id = meta_preds[:model.num_colors].argmax().item()
#     pred_category_id = meta_preds[model.num_colors:model.num_colors + model.num_categories].argmax().item()
#     pred_object_logits = meta_preds[model.num_colors + model.num_categories:]
#     pred_object_id = pred_object_logits.argmax().item()

#     # 3. Initialization: Beams list stores (sequence_ids, cumulative_log_probability)
#     initial_seq = [SOS_ID]
#     beams = [(initial_seq, 0.0)]
    
#     # 4. Beam Search Loop
#     for _ in range(max_len):
#         new_beams = []
        
#         for seq, score in beams:
#             if seq[-1] == EOS_ID:
#                 new_beams.append((seq, score))
#                 continue
                
#             input_ids = torch.tensor([seq], dtype=torch.long, device=device)

#             logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
#             next_token_logits = logits[:, -1, :].squeeze(0)
            
#             # Use log probabilities for stability
#             log_probs = F.log_softmax(next_token_logits, dim=-1)
#             top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
#             for k in range(beam_width):
#                 new_token_id = top_ids[k].item()
#                 new_log_prob = top_log_probs[k].item()
                
#                 new_seq = seq + [new_token_id]
#                 new_score = score + new_log_prob 
                
#                 new_beams.append((new_seq, new_score))

#         # Select the top 'beam_width' candidates
#         beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
#         if beams[0][0][-1] == EOS_ID:
#             break
            
#     # 5. Final Output Selection
#     best_seq = beams[0][0]
    
#     if best_seq[-1] == EOS_ID:
#         predicted_text_ids = best_seq[1:-1]
#     else:
#         predicted_text_ids = best_seq[1:]
        
#     predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    
#     return predicted_text, pred_color_id, pred_category_id, pred_object_id


@torch.no_grad()
def generate_text_beam_predicted_meta(model, eeg_signal, edge_index, edge_attr, 
                                      beam_width=5, max_len=64):
    """
    Generates text using Beam Search, conditioning the decoder 
    on the METADATA PREDICTED by the model's head from the EEG CLS token.
    
    Args:
        model (nn.Module): The trained Seq2SeqTF model.
        eeg_signal (Tensor): Single EEG sample (C, T).
        edge_index (Tensor): Granger causality graph edges (static).
        edge_attr (Tensor): Granger causality graph attributes (static).
        beam_width (int): Beam search width.
        max_len (int): Maximum sequence length.

    Returns:
        tuple: (predicted_text, pred_color_id, pred_category_id, pred_object_id)
    """
    model.eval()
    
    # Unsqueeze input to create a batch size of 1
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    cls_token_feature = eeg_features[:, 0, :]
    
    # 2. Predict Metadata (The new, crucial step)
    meta_preds = model.meta_head(cls_token_feature) # (1, D_meta_output)
    
    # Extract predicted IDs/features
    # Note: Use argmax for categorical IDs and the raw logits for object features
    pred_color_logits = meta_preds[:, :model.num_colors]
    pred_category_logits = meta_preds[:, model.num_colors:model.num_colors + model.num_categories]
    pred_object_logits = meta_preds[:, model.num_colors + model.num_categories:]

    pred_color_id = pred_color_logits.argmax(dim=1)
    pred_category_id = pred_category_logits.argmax(dim=1)
    pred_object_id = pred_object_logits.argmax(dim=1) # Used for reporting/debugging

    # 3. Reconstruct 'meta_signal' and Encode Predicted Metadata
    # The MetadataEncoder expects [Color ID, Category ID, Object Features]
    predicted_meta_input = torch.cat([
        pred_color_id.unsqueeze(1).float(), 
        pred_category_id.unsqueeze(1).float(),
        pred_object_logits.float() # Using logits as the feature vector
    ], dim=1) 
    
    # Encode the reconstructed input to get the conditional features
    predicted_meta_features = model.meta_encoder(predicted_meta_input) # (1, D_meta)
    
    decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
    memory_mask = None 

    # 4. Initialization: Beams list stores (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 5. Beam Search Loop (Conditioning on PREDICTED metadata features)
    for _ in range(max_len):
        new_beams = []
        
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            input_ids = torch.tensor([seq], dtype=torch.long, device=device)

            # CRITICAL CHANGE: Use predicted_meta_features here!
            logits = model.decoder(input_ids, decoder_memory, memory_mask, predicted_meta_features) 
            next_token_logits = logits[:, -1, :].squeeze(0)
            
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob 
                
                new_beams.append((new_seq, new_score))

        # Select the top 'beam_width' candidates
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        if beams[0][0][-1] == EOS_ID:
            break
            
    # 6. Final Output Selection
    best_seq = beams[0][0]
    
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    
    # Return single-item IDs for consistency with the original function's output
    return (predicted_text, 
            pred_color_id.item(), 
            pred_category_id.item(), 
            pred_object_id.item())
# --- 6. Final Evaluation Execution ---
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model.pt'

# Load the best weights (assuming model structure 'new_model' is defined)
try:
    new_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Successfully loaded best model weights from: {MODEL_SAVE_PATH}")
except Exception as e:
    print(f"Error loading model weights: {e}. Cannot guarantee final results.")

predictions_list = []
references_list = []
samples_printed = 0
NUM_SAMPLES_TO_PRINT = 10 
global_sample_index = 0

bleu_metric = evaluate.load('bleu')
test_progress_bar = tqdm(globals()['test_loader'], desc="Beam Search Evaluation (Beam 5)", leave=True)
start_time = time.time()

print("\n--- Starting Beam Search Evaluation (Qualitative & Quantitative) ---")

for eeg_b, meta_b, txt_b in test_progress_bar:
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i]
        true_text_ids = txt_b[i]
        
        # --- Generation & Prediction ---
        predicted_text, pred_color_id, pred_category_id, pred_object_id = generate_text_beam_predicted_meta(
            new_model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr, beam_width=5
        )
        
        # --- Ground Truth Decoding and Metadata Extraction ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        true_color_id = int(meta_sample[0].item())
        true_category_id = int(meta_sample[1].item())
        
        if not true_text: continue
            
        # Store for Corpus-Level Evaluation
        predictions_list.append(predicted_text)
        references_list.append([true_text])

        # --- Print Sample Analysis (Qualitative & Metadata Check) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            
            # Decode names using the global maps (falls back to ID if not found)
            pred_color_name = id_to_color.get(pred_color_id, f"ID: {pred_color_id}")
            true_color_name = id_to_color.get(true_color_id, f"ID: {true_color_id}")
            pred_category_name = id_to_scene.get(pred_category_id, f"ID: {pred_category_id}")
            true_category_name = id_to_scene.get(true_category_id, f"ID: {true_category_id}")
            pred_object_name = object_mapping.get(str(pred_object_id), f"ID: {pred_object_id}")
            
            true_object_indices = meta_sample[2:].nonzero(as_tuple=True)[0]
            true_object_names = [object_mapping.get(str(idx.item()), f"ID: {idx.item()}") for idx in true_object_indices]
            if not true_object_names: true_object_names = ["None"]

            print(f"\n--- Sample {global_sample_index + 1} (Beam 5) ---")
            print(f"GROUND TRUTH TEXT: {true_text}")
            print(f"PREDICTED TEXT:    {predicted_text}")
            print("\n  --- METADATA PREDICTION (from EEG CLS) ---")
            print(f"  Color:      Truth='{true_color_name}' | Pred='{pred_color_name}'")
            print(f"  Category:   Truth='{true_category_name}' | Pred='{pred_category_name}'")
            print(f"  Object(s):  Truth={', '.join(true_object_names)} | Pred='{pred_object_name}'")
            samples_printed += 1
            
        global_sample_index += 1

# --- 7. Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Beam Search Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score (Beam 5):** {final_bleu_score:.4f} |")
print("=============================================")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Error loading model weights: name 'new_model' is not defined. Cannot guarantee final results.


KeyError: 'test_loader'